In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss
from sklearn.preprocessing import LabelEncoder

In [2]:
class MLkNN:
    def __init__(self, k=10, s=1.0):
        self.k = k
        self.s = s
 
    def fit(self, X, y):
        self.y_train = y
        self.knn = NearestNeighbors(n_neighbors=self.k).fit(X)
        L = y.shape[1]
        s, k = self.s, self.k
 
        # Priori P(H1) e P(H0)
        self.ph1 = (s + y.sum(axis=0)) / (s * 2 + len(y))
        self.ph0 = 1 - self.ph1
 
        # Verossimilhança P(E_j | H_b)
        self.cond1 = np.zeros((L, k + 1))
        self.cond0 = np.zeros((L, k + 1))
 
        neighbors = self.knn.kneighbors(X, return_distance=False)
        for i in range(len(X)):
            delta = self.y_train[neighbors[i]].sum(axis=0).astype(int)
            for l in range(L):
                if y[i, l] == 1:
                    self.cond1[l, delta[l]] += 1
                else:
                    self.cond0[l, delta[l]] += 1
 
        # Suavização de Laplace
        self.cond1 = (self.cond1 + s) / (self.cond1.sum(axis=1, keepdims=True) + s * (k + 1))
        self.cond0 = (self.cond0 + s) / (self.cond0.sum(axis=1, keepdims=True) + s * (k + 1))
        return self
 
    def predict(self, X):
        neighbors = self.knn.kneighbors(X, return_distance=False)
        preds = []
        for i in range(len(X)):
            delta = self.y_train[neighbors[i]].sum(axis=0).astype(int)
            L = self.y_train.shape[1]
            row = []
            for l in range(L):
                p1 = self.ph1[l] * self.cond1[l, delta[l]]
                p0 = self.ph0[l] * self.cond0[l, delta[l]]
                row.append(1 if p1 >= p0 else 0)
            preds.append(row)
        return np.array(preds)

In [3]:
# carregando o conjunto de dados visto em sala de aula (sapos)
df = pd.read_csv("amphibians.csv", sep=";", header=1)
 
label_names = [
    "Green frogs", "Brown frogs", "Common toad",
    "Fire-bellied toad", "Tree frog", "Common newt", "Great crested newt"
]
 
X_df = df.iloc[:, 1:-7]

for col in X_df.columns:
    if X_df[col].dtype == object:
        X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

X = X_df.values
y = df.iloc[:, -7:].values

In [4]:
print("=" * 55)
print("Dataset Amphibians — Estrutura")
print("=" * 55)
print(f"Instâncias : {X.shape[0]}")
print(f"Features   : {X.shape[1]}")
print(f"Rótulos    : {y.shape[1]}")
print(f"\nDistribuição dos rótulos:")
for i, name in enumerate(label_names):
    freq = y[:, i].mean() * 100
    print(f"  {name:<22}: {freq:.1f}% positivos")

Dataset Amphibians — Estrutura
Instâncias : 189
Features   : 15
Rótulos    : 7

Distribuição dos rótulos:
  Green frogs           : 57.1% positivos
  Brown frogs           : 78.3% positivos
  Common toad           : 65.6% positivos
  Fire-bellied toad     : 30.7% positivos
  Tree frog             : 37.6% positivos
  Common newt           : 30.7% positivos
  Great crested newt    : 11.1% positivos


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [6]:
classifier = MLkNN(k=10, s=1.0)
classifier.fit(X_train, y_train)

In [7]:
y_pred = classifier.predict(X_test)

In [8]:
hl = hamming_loss(y_test, y_pred)

In [9]:
print(f"Hamming Loss : {hl:.4f}  ({hl*100:.2f}% de rótulos errados)")

Hamming Loss : 0.2807  (28.07% de rótulos errados)
